# PyRPL spectrum-analyzer curves &mdash; convert &amp; plot

This notebook is a thin driver around two scripts that must sit **next to this
notebook**:

* `dat_to_csv.py` &mdash; converts PyRPL `.dat` curves to CSV
* `plot_curve.py` &mdash; plots / overlays / merges the CSVs

All the real work lives in those files; the notebook just imports their
functions and calls them. PyRPL saves **one channel per file**, so the GUI's
three traces (in1 = blue, in2 = green, cross = magenta) are three separate CSVs.


## 1. Setup

In [ ]:
import os, sys, glob

# make sure the two scripts (in the notebook's folder) are importable
sys.path.insert(0, os.getcwd())
import dat_to_csv          # provides load_dat(), convert(), main()
import plot_curve          # provides read_header(), main(), ...

%matplotlib inline

# ---- edit these -----------------------------------------------------------
CURVE_DIR = os.path.expanduser("~/pyrpl_user_dir/curve")   # folder with .dat files
CSV_DIR   = os.path.join(CURVE_DIR, "csv")                 # where .csv are written
RECONVERT = False     # True -> re-convert even if CSVs already exist
# ---------------------------------------------------------------------------


def run_convert(in_dir=CURVE_DIR, out_dir=CSV_DIR):
    """Call dat_to_csv.main() with the right argv (reuses the script as-is)."""
    old = sys.argv
    sys.argv = ["dat_to_csv.py", in_dir, out_dir]
    try:
        dat_to_csv.main()
    except SystemExit as e:
        print(e)
    finally:
        sys.argv = old


def run_plot(paths, save=None, combine=None):
    """Call plot_curve.main() with the right argv (reuses the script as-is)."""
    argv = ["plot_curve.py", *paths]
    if combine:
        argv += ["--combine", combine]
    if save:
        argv.append(save)
    old = sys.argv
    sys.argv = argv
    try:
        plot_curve.main()
    except SystemExit as e:
        print(e)
    finally:
        sys.argv = old


os.makedirs(CSV_DIR, exist_ok=True)
print("CURVE_DIR:", CURVE_DIR)
print(".dat files found:", len(glob.glob(os.path.join(CURVE_DIR, "*.dat"))))


## 2. Convert `.dat` &rarr; `.csv`

Runs `dat_to_csv.main()`; skips conversion if the CSVs already exist (set `RECONVERT = True` to force).

In [ ]:
dats = sorted(glob.glob(os.path.join(CURVE_DIR, "*.dat")),
              key=lambda p: (len(os.path.basename(p)), p))
expected = [os.path.join(CSV_DIR, os.path.splitext(os.path.basename(p))[0] + ".csv")
            for p in dats]

if RECONVERT or not all(os.path.exists(c) for c in expected):
    run_convert()
else:
    print("All CSVs already present in", CSV_DIR, "(skipping; set RECONVERT=True to redo)")

csv_paths = sorted(glob.glob(os.path.join(CSV_DIR, "*.csv")),
                   key=lambda p: (len(os.path.basename(p)), p))
print("CSV files:", [os.path.basename(p) for p in csv_paths])


## 3. Inspect channel names

Uses `plot_curve.read_header` to show each file's `# name:` so you can pick the in1 / in2 / cross trio.

In [ ]:
for p in csv_paths:
    h = plot_curve.read_header(p)
    print(f"{os.path.basename(p):10s}  name={h.get('name', h.get('curve_name', '?'))}")


## 4. Plot a chosen set of channels

Set `FILES` to the CSVs you want to overlay (filenames inside `CSV_DIR`).

In [ ]:
FILES = [os.path.basename(p) for p in csv_paths[:3]]   # edit to your trio
paths = [os.path.join(CSV_DIR, f) for f in FILES]
print("plotting:", FILES)
run_plot(paths)


## 5. Overlay every CSV in the folder

In [ ]:
run_plot(csv_paths)


## 6. Merge the chosen channels into one wide CSV

Writes `combined.csv` (one frequency column + every channel's columns) and plots it &mdash; the *one CSV, several traces* view.

In [ ]:
combined = os.path.join(CSV_DIR, "combined.csv")
run_plot(paths, combine=combined)        # merges + shows
print("\nplotting the merged single CSV on its own:")
run_plot([combined])
